In [8]:
!nvidia-smi

Fri Apr 25 04:57:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
import nltk
from nltk.corpus import stopwords, gutenberg
from nltk.stem import WordNetLemmatizer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F  # For one-hot vectors
from collections import Counter

In [10]:
nltk.download('stopwords')
nltk.download('gutenberg')  # Corrected spelling
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package gutenberg to /usr/share/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
text = gutenberg.raw('austen-emma.txt')

In [14]:
import re
stopwords_list = stopwords.words('english')
wn = WordNetLemmatizer()
def preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    no_stopwords = [word for word in text.split() if word not in stopwords_list]
    preprocessed = ' '.join([wn.lemmatize(word) for word in no_stopwords])
    return preprocessed

In [15]:
preprocessed_text=preprocessing(text)

In [16]:
len(preprocessed_text)

503588

In [17]:
words=[word for word in preprocessed_text.split()]
len(words)

73292

In [23]:
word_counts = Counter(words)
vocab = [word for word, freq in word_counts.items() if freq > 5]
word2index = {word: index for index, word in enumerate(vocab)}
index2word = {index: word for word, index in word2index.items()}

In [24]:
word2index['play']

1428

In [25]:
words=[word for word in words if word in word2index]

In [26]:
len(words)

64518

In [31]:
window_size = 2
dataset = []
for i, central_word in enumerate(words):
    central_word_index = word2index[central_word]
    for j in range(-window_size, window_size + 1):
        if j != 0 and 0 <= i + j < len(words):
            context_word_index = word2index[words[i + j]]
            dataset.append((central_word_index, context_word_index))

In [32]:
central_word_torch=torch.tensor([pair[0] for pair in dataset])
context_word_torch=torch.tensor([pair[1] for pair in dataset])

In [33]:
torch_dataset=TensorDataset(central_word_torch,context_word_torch)
train_loader=DataLoader(torch_dataset,batch_size=64,shuffle=True)

In [47]:
class Word2VecScratch(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Word2VecScratch, self).__init__()
        seld.hidden = nn.Linear(vocal_size, embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)
    def forward(self, x):
        x = self.hidden(x)
        x = self.output(x)
        return x

In [35]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model=Word2VecScratch(len(vocab),300).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=3e-4)

cuda


In [36]:
epochs=50
for i in range(epochs):
  loss=0
  model.train()
  for central_word,context_word in train_loader:
    central_word=central_word.to(device)
    context_word=context_word.to(device)
    central_word_one_hot=F.one_hot(central_word,num_classes=len(vocab)).float().to(device)
    context_word_one_hot=F.one_hot(context_word,num_classes=len(vocab)).float().to(device)
    outputs=model(central_word_one_hot)
    batch_loss=loss_fn(outputs,context_word_one_hot)
    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()
    loss+=batch_loss.item()
  print(f'Epoch: {i+1} Training_Loss:{loss/len(train_loader)}')

Epoch: 1 Training_Loss:6.692761568641237
Epoch: 2 Training_Loss:6.598535241898609
Epoch: 3 Training_Loss:6.541535696024461
Epoch: 4 Training_Loss:6.468278877911898
Epoch: 5 Training_Loss:6.386429981058595
Epoch: 6 Training_Loss:6.300588145375459
Epoch: 7 Training_Loss:6.214392795652602
Epoch: 8 Training_Loss:6.130131811827251
Epoch: 9 Training_Loss:6.049517050529572
Epoch: 10 Training_Loss:5.973427184781453
Epoch: 11 Training_Loss:5.902214389624148
Epoch: 12 Training_Loss:5.836349915429495
Epoch: 13 Training_Loss:5.775690038062904
Epoch: 14 Training_Loss:5.720190779730666
Epoch: 15 Training_Loss:5.669466267204332
Epoch: 16 Training_Loss:5.623016076379989
Epoch: 17 Training_Loss:5.581169590357661
Epoch: 18 Training_Loss:5.543041802445801
Epoch: 19 Training_Loss:5.508653703647484
Epoch: 20 Training_Loss:5.477331844772501
Epoch: 21 Training_Loss:5.448997070970826
Epoch: 22 Training_Loss:5.423573009198693
Epoch: 23 Training_Loss:5.400636682195874
Epoch: 24 Training_Loss:5.379841251288163
E

In [37]:
word_embeddings=model.hidden.weight.detach()

In [38]:
word_embeddings.shape

torch.Size([300, 1815])

In [40]:
word_embeddings=word_embeddings.T

In [43]:
from scipy.spatial import distance
distance = distance.cosine(word_embeddings[0].cpu().numpy(),word_embeddings[1].cpu().numpy())
print(distance)

0.9673314771226549
